# 03 · CANN: nesting the GLM in a network
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/03_cann.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/actuarialsociety/deep-actuarial-science/blob/main/notebooks/03_cann.ipynb) [![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/03_cann.ipynb)

$\lambda^{\mathrm{CANN}}(x)=\lambda^{\mathrm{GLM}}(x)\cdot\exp\{w_0+\langle w,z^{(d:1)}(x)\rangle\}$, initialised at $w=0$: training starts exactly at the GLM.

_Runs anywhere: **Colab** opens on a GPU runtime (T4) and the first cell stops on a CPU runtime; on **Kaggle** turn on *GPU T4* and *Internet* in the notebook settings; on **SageMaker Studio Lab** start a GPU runtime. In Azure ML, SageMaker Studio or any other Jupyter, clone the repo (or upload this notebook) and run all: the first cell installs what it needs._

In [ ]:
# Setup: runs anywhere Jupyter runs. Opened on its own (Colab, Kaggle, SageMaker, Azure ML,
# any hosted Jupyter), this cell clones and installs the repo; in a clone or the container it
# uses that checkout.
import os, sys, importlib.util
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

REPO_DIR = next((d for d in (os.getcwd(), os.path.dirname(os.getcwd()))
                 if os.path.isdir(os.path.join(d, "src", "deep_actuarial_science"))), None)
if REPO_DIR is None:
    if sys.version_info < (3, 10):
        raise RuntimeError("Needs Python 3.10+: choose a newer kernel or image")
    # a writable, persistent home: Colab, Kaggle, SageMaker notebook instances, else ~
    base = next(d for d in ("/content", "/kaggle/working", os.path.expanduser("~/SageMaker"),
                            os.path.expanduser("~")) if os.path.isdir(d) and os.access(d, os.W_OK))
    REPO_DIR = os.path.join(base, "deep-actuarial-science")
    if not os.path.exists(REPO_DIR):  # Kaggle: turn on Internet in the notebook settings
        !git clone -q https://github.com/actuarialsociety/deep-actuarial-science {REPO_DIR}
    if importlib.util.find_spec("deep_actuarial_science") is None:
        !pip install -q -e {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui, device_summary
device_summary()
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
run("frequency", models=["glm3", "cann"])
leaderboard("frequency", ["glm3", "cann"])

In [ ]:
load("cann.freq_reconciliation")

## Open the box: zero initialisation reproduces the GLM
Build a CANN with 0 epochs: its predictions equal the GLM's to float precision.

In [ ]:
import numpy as np
from deep_actuarial_science.models.frequency import CANN
glm, test = load("glm.freq_model"), load("freq_test")
cann0 = CANN().fit(load("freq_learn"), {"epochs": 0}, base=glm)
np.abs(cann0.predict(test) / glm.predict(test) - 1).max()

## Where does the network correct the GLM?
The ratio CANN / GLM by covariate shows what the GLM misspecifies (e.g. the young-driver and bonus-malus shapes).

In [ ]:
p = load("cann.freq_predictions").assign(glm=load("glm.freq_predictions").pred)
p = p[p.split == "test"]
r = p.groupby(p.DrivAge.clip(upper=90))[["pred", "glm"]].sum()
(r.pred / r.glm).plot(title="CANN / GLM by driver age", ylabel="multiplicative correction");

In [ ]:
show("frequency", "cann", ["calibration", "marginal_drivage", "training"])

## Exercise
Freeze less: make the GLM part trainable (hint: subclass `CANN` and make the offset a learnable linear layer). Does joint training beat the offset version?

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. MLflow runs on the notebook's machine: in Colab it opens below and in a new tab through Colab's proxy; in SageMaker and Azure ML through Jupyter's proxy; on Kaggle, which exposes no ports, query the runs with `mlflow.search_runs` instead.

In [ ]:
mlflow_ui()